# Import modules and scripts

In [2]:
import sys, os

mod_dir = os.path.abspath('/Users/rangulo/projects/EtaCar_LEs/python')
sys.path.append(mod_dir)

from LC_utility import *

In [3]:
%load_ext autoreload
%autoreload 2

# Read in data

In [4]:
### Read in data that has been flagged for good indices

lesdf_a57g, lcdfs_a57g = readin_LCdfs('LC_tables/modified_data/LCs_a57g.h5')
lesdf_a57r, lcdfs_a57r = readin_LCdfs('LC_tables/modified_data/LCs_a57r.h5')
lesdf_a57i, lcdfs_a57i = readin_LCdfs('LC_tables/modified_data/LCs_a57i.h5')
lesdf_a57z, lcdfs_a57z = readin_LCdfs('LC_tables/modified_data/LCs_a57z.h5')

lesdf_a54g, lcdfs_a54g = readin_LCdfs('LC_tables/modified_data/LCs_a54g.h5')
lesdf_a54r, lcdfs_a54r = readin_LCdfs('LC_tables/modified_data/LCs_a54r.h5')
lesdf_a54i, lcdfs_a54i = readin_LCdfs('LC_tables/modified_data/LCs_a54i.h5')
lesdf_a54z, lcdfs_a54z = readin_LCdfs('LC_tables/modified_data/LCs_a54z.h5')

## Save to dict for convenience
lesdf_a57 = {'g': lesdf_a57g, 'r': lesdf_a57r, 'i': lesdf_a57i, 'z': lesdf_a57z}
lcdfs_a57 = {'g': lcdfs_a57g, 'r': lcdfs_a57r, 'i': lcdfs_a57i, 'z': lcdfs_a57z}

lesdf_a54 = {'g': lesdf_a54g, 'r': lesdf_a54r, 'i': lesdf_a54i, 'z': lesdf_a54z}
lcdfs_a54 = {'g': lcdfs_a54g, 'r': lcdfs_a54r, 'i': lcdfs_a54i, 'z': lcdfs_a54z}

In [5]:
### Exclude 2003 image -- messes up fit
# for key in lcdfs_a54i.keys():
#     df_54i = lcdfs_a54i[key]
#     excl_a54i = np.where(df_54i['fitsfile'].eq('ec0915a54.i.030310.CT4m.x0028.a0_160305.523435_ooi_i_v1_N22.diff.fits'))[0]
#     df_54i.loc[excl_a54i, 'g_ixs_flag'] = False

# for key in lcdfs_a57i.keys():
#     df_57i = lcdfs_a57i[key]
#     excl_a57i = np.where(df_57i['fitsfile'].eq('ec0915.i.030310.CT4m.x0028.a0_160305.523435_ooi_i_v1_N26.diff.fits'))[0]
#     df_57i.loc[excl_a57i, 'g_ixs_flag'] = False

In [6]:
### For defining fit runs

save_nm = 'test6'

### Exclude older images
excl_older = False
# excl_older = True

### Skip first part -- for troubleshooting
skip_1st = False
# skip_1st = True

### Skip segment fitting -- for troubleshooting
skip_segm = False
# skip_segm = True

### Fit gp of segments (median curve)
gp_segm = False
# gp_segm = True

In [7]:
### Define save directory
save_dir = f"fitting_results/{save_nm}"

if excl_older: save_dir = f"{save_dir}/no_older"

subdirs = ['samples', 'fitted_params', 'final_data']
for sdir in subdirs:
    spath = os.path.join(save_dir, sdir)
    os.makedirs(spath, exist_ok=True)


### Include or exclude older reductions
if excl_older:
    for fband in lcdfs_a54.keys():
        dfs = lcdfs_a54[fband]
        for key in dfs.keys():
            df = dfs[key]
            excl_inds = np.where((~df['im_instrum'].eq('DECam')) & (~df['im_instrum'].eq('DECam(badband-corrected)')))[0]
            df.loc[excl_inds, 'g_ixs_flag'] = False

    for fband in lcdfs_a57.keys():
        dfs = lcdfs_a57[fband]
        for key in dfs.keys():
            df = dfs[key]
            excl_inds = np.where((~df['im_instrum'].eq('DECam')) & (~df['im_instrum'].eq('DECam(badband-corrected)')))[0]
            df.loc[excl_inds, 'g_ixs_flag'] = False

# Fitting Process

## Fit positions

In [8]:
### Data divided into groups based on positons/phase
    ## a57 groups: 0 - main
    ## a54 groups: 0 - tail end (comet), 1 - main peak, 2 - plateau


le_group1 = 0
tmpl_ID1 = 'lc_0_7'

le_group2 = 1
tmpl_ID2 = 'lc_1_55'

le_group3 = 2
tmpl_ID3 = 'lc_2_93'

le_group4 = 0
tmpl_ID4 = 'lc_0_23'

In [9]:
### Fit light curves to one position (one light curve) per segment per filter

chain_sz1 = 5000
per_key = True
peakfilt = 'i'
fixed_tmpl = None

if not skip_1st:
    mcres_1 = emcee_fit(lcdfs_a57, le_group=le_group1, tmpl_key=tmpl_ID1, peak_range='full', peak_filt=peakfilt, fixed_params=fixed_tmpl, per_key_fit=per_key, chain_sz=chain_sz1, outputfile=f"{save_dir}/samples/1stsamples_1.h5")
    mcres_2 = emcee_fit(lcdfs_a54, le_group=le_group2, tmpl_key=tmpl_ID2, peak_range='full', peak_filt=peakfilt, fixed_params=fixed_tmpl, per_key_fit=per_key, chain_sz=chain_sz1, outputfile=f"{save_dir}/samples/1stsamples_1.h5")
    mcres_3 = emcee_fit(lcdfs_a54, le_group=le_group3, tmpl_key=tmpl_ID3, peak_range=[57300, 58000], peak_filt=peakfilt, fixed_params=fixed_tmpl, per_key_fit=per_key, chain_sz=chain_sz1, outputfile=f"{save_dir}/samples/1stsamples_1.h5")
    mcres_4 = emcee_fit(lcdfs_a54, le_group=le_group4, tmpl_key=tmpl_ID4, peak_range=None, peak_filt=peakfilt, fixed_params=fixed_tmpl, per_key_fit=per_key, chain_sz=chain_sz1, outputfile=f"{save_dir}/samples/1stsamples_1.h5")
else:
    ### Read in previous run
    mcres_1 = load_fitparams(f"{save_dir}/fitted_params/1stparams_1.h5")
    mcres_2 = load_fitparams(f"{save_dir}/fitted_params/1stparams_2.h5")
    mcres_3 = load_fitparams(f"{save_dir}/fitted_params/1stparams_3.h5")
    mcres_4 = load_fitparams(f"{save_dir}/fitted_params/1stparams_4.h5")

In [17]:
### Combine aligned light curves into one data frame per segment per filter

combs1 = get_combLCs(lcdfs_a57, mcres_1)
combs2 = get_combLCs(lcdfs_a54, mcres_2)
combs3 = get_combLCs(lcdfs_a54, mcres_3)
combs4 = get_combLCs(lcdfs_a54, mcres_4)
combs_ls = [combs1, combs2, combs3, combs4]

combs_dict = {}
for fband in combs1.keys():
    combs_dict[fband] = {}
    for i, cb in enumerate(combs_ls):
        combs_dict[fband][f'seg{i+1}'] = cb[fband]

In [21]:
### Three-sigma cuts

gps_dict = {}

for fband in combs_dict.keys():
    gps_dict[fband] = {}
    cbs = combs_dict[fband]
    for key, cb in cbs.items():
        if fband == 'i':
            excl_inds = np.where(cb['fitsfile'].eq('ec0915a54.i.030310.CT4m.x0028.a0_160305.523435_ooi_i_v1_N22.diff.fits') | 
                                 cb['fitsfile'].eq('ec0915.i.030310.CT4m.x0028.a0_160305.523435_ooi_i_v1_N26.diff.fits'))[0]
        else: excl_inds = None
        sigmacut_df(cb, sigma=3.0, exclude_inds=excl_inds)

        gps_dict[fband][key] = optimizeGP_celer(cb, lc_key=None, flag='sigmaclip', exclude_inds=excl_inds)

### Plots

In [32]:
### Check data

# dfs = lcdfs_a54['i']
# le_grp = getLC_grp(dfs, 0)

# for i, key in enumerate(le_grp):
#     # if key=='0.1':
#     print(key)
#     df = dfs[key]
#     inds_sci = getLC_inds(dfs, lc_key=key, flag='g_ixs')
#     # sel_inds = np.where(~df.t['im_instrum'].eq('DECam') & ~df.t['im_instrum'].eq('DECam(badband-corrected)'))[0]
#     # inds_sci = AandB(inds_sci, sel_inds)

#     t_sci, f_sci, err_sci = getLC_vals(dfs, lc_key=key, indices=inds_sci)

#     plt.errorbar(t_sci,f_sci,yerr=err_sci, fmt='.')
    
#     instrum_l = df.loc[inds_sci,'im_instrum']

#     for inst in np.unique(instrum_l):
#         ims_inds = np.where(instrum_l.eq(inst))[0]

#         plt.errorbar(t_sci[ims_inds],f_sci[ims_inds],yerr=err_sci[ims_inds], fmt='.', label=inst)
#     plt.legend()
#     plt.show()


# # plot_LCs(dfs)

In [29]:
### Check normalizations

# for key in mcres_1['sel_keys']:
#     plt.scatter(mcres_1[f'anorm_{key}'], 1-mcres_1[f'dEgi_{key}'])

# plt.xlabel('anorm_dust')
# plt.ylabel('Color (g-i)')

In [24]:
### See fits

# plot_LCfits(lcdfs_a57, mcres_1)

In [26]:
### Check corner plots

# ch_samps = load_samples(f"{save_dir}/samples/1stsamples_1.h5")
# # fband = 'i'
# for key in ch_samps.keys():
#     check = corner.corner(ch_samps[key].values, labels=ch_samps[key].keys(), quantiles=[0.5])
#     plt.show()

In [33]:
## Plot segments

# filt_s = 'i'
# for fband, cbs in combs_dict.items():
#     if fband == filt_s:
#         for seg, cb in cbs.items():
#             plt.errorbar(cb['mjd'], cb['muJyas2'], yerr=cb['muJyas2_err'], fmt='.')
#             inds = np.where(cb['sigmaclip_flag'].eq(False))[0]
#             plt.errorbar(cb.loc[inds,'mjd'], cb.loc[inds,'muJyas2'], yerr=cb.loc[inds,'muJyas2_err'], fmt='.')
#             plt.plot(gps_dict[fband][seg]['mjd'], gps_dict[fband][seg]['muJyas2'], color='k', zorder=5)
#             plt.show()

## Fit segments

In [39]:
tmpl_test = combs_dict['i']['seg2']
sci_test = combs_dict['i']['seg1']

In [40]:
sci_test

,mjd,muJyas2,muJyas2_err,orig_key,orig_ind,im_instrum,fitsfile,sigmaclip_flag
0,52406.285708,-1.668291,0.653410,lc_0_12,0,CT4m,ec0915.i.030310.CT4m.x0028.a0_160305.523435_oo...,False
1,52413.371345,-0.021439,0.954143,lc_0_11,0,CT4m,ec0915.i.030310.CT4m.x0028.a0_160305.523435_oo...,False
2,52413.946088,-2.451248,1.329265,lc_0_13,0,CT4m,ec0915.i.030310.CT4m.x0028.a0_160305.523435_oo...,False
3,52426.269808,-1.038498,0.968735,lc_0_1,0,CT4m,ec0915.i.030310.CT4m.x0028.a0_160305.523435_oo...,False
4,52517.997247,-1.890502,0.965493,lc_0_14,0,CT4m,ec0915.i.030310.CT4m.x0028.a0_160305.523435_oo...,False
...,...,...,...,...,...,...,...,...
4714,61214.508972,5.673608,0.378930,lc_0_10,435,DECam,ec0915.260406.1471988_ooi_i_v1_N26_160305.5234...,False
4715,61215.439633,5.203675,0.287088,lc_0_10,436,DECam,ec0915.260407.1472318_ooi_i_v1_N26_160305.5234...,False
4716,61216.243925,5.005073,0.280281,lc_0_10,437,DECam,ec0915.260408.1472678_ooi_i_v1_N26_160305.5234...,False
4717,61217.443509,5.063814,0.296319,lc_0_10,438,DECam,ec0915.260409.1473230_ooi_i_v1_N26_160305.5234...,False


In [ ]:
### Fit segments together per filter 
    ### Individual vs Simulataneous

if not skip_segm:
    if not gp_segm:
        segms = combs_dict
        fflag = 'sigmaclip'
    else:
        segms = gps_dict
        fflag = None

    chain_sz2 = 5000
    tshift_34 = -4450

    tmpl_seg = 'seg2'
    tmpl_range = {'seg2': 'full'}

    mcres_seg = emcee_fit(segms, le_group=None, tmpl_key=tmpl_seg, peak_range='full', peak_filt=peakfilt, fixed_params=fixed_tmpl, per_key_fit=per_key, chain_sz=chain_sz1, outputfile=f"{save_dir}/samples/segm_samples.h5")

#     if fit_method == 'individual':
#         mcres_21 = loop_fitsegdata(segms2, segms1, peak_range1='full', peak_range2='full', peak_filt=peakfilt, 
#                                 unify_norm=unifynorm, linearsolve=linear_solve, norm_anchor=anchorn, flag=fflag, 
#                                 outputfile=f"{save_dir}/samples/1stsamples_21.h5", chain_sz=chain_sz2)
#         new_segms = comb_combs([segms1, segms2], [mcres_21, None])

#         mcres_23 = loop_fitsegdata(new_segms, segms3, peak_range1=[59100,59500], peak_range2=[57500,58000], peak_filt=peakfilt, 
#                                 unify_norm=unifynorm, linearsolve=linear_solve, norm_anchor=anchorn, flag=fflag, 
#                                 outputfile=f"{save_dir}/samples/1stsamples_23.h5", chain_sz=chain_sz2)
#         new_segms = comb_combs([new_segms, segms3], [None, mcres_23])

#         mcres_34 = loop_fitsegdata(new_segms, segms4, tshift_init=tshift_34, peak_filt=peakfilt, 
#                                 unify_norm=unifynorm, linearsolve=linear_solve, norm_anchor=anchorn, flag=fflag, 
#                                 outputfile=f"{save_dir}/samples/1stsamples_34.h5", chain_sz=chain_sz2)


#     elif fit_method == 'simultaneous':
#         mcres_21 = fit_segdata(segms2, segms1, peak_range1='full', peak_range2='full', peak_filt=peakfilt, 
#                             local_fparams=local_fparams, unify_norm=unifynorm, linearsolve=linear_solve, norm_anchor=anchorn, flag=fflag, 
#                             outputfile=f"{save_dir}/samples/1stsamples_21.h5", chain_sz=chain_sz2)
#         new_segms = comb_combs([segms1, segms2], [mcres_21, None])

#         mcres_23 = fit_segdata(new_segms, segms3, peak_range1=[59100,59500], peak_range2=[57500,58000], peak_filt=peakfilt, 
#                             local_fparams=local_fparams, unify_norm=unifynorm, linearsolve=linear_solve, norm_anchor=anchorn, flag=fflag, 
#                             outputfile=f"{save_dir}/samples/1stsamples_23.h5", chain_sz=chain_sz2)
#         new_segms = comb_combs([new_segms, segms3], [None, mcres_23])

#         mcres_34 = fit_segdata(new_segms, segms4, tshift_init=tshift_34, peak_filt=peakfilt, 
#                             local_fparams=local_fparams, unify_norm=unifynorm, linearsolve=linear_solve, norm_anchor=anchorn, flag=fflag, 
#                             outputfile=f"{save_dir}/samples/1stsamples_34.h5", chain_sz=chain_sz2)


#     ### Save to dict for convenience
#     segm_mcres = {'21': mcres_21, '23': mcres_23, '34': mcres_34}


# else:
#     segm_mcres = load_fitparams(f"{save_dir}/fitted_params/segm_params.h5")
#     mcres_21 = segm_mcres['21']
#     mcres_23 = segm_mcres['23']
#     mcres_34 = segm_mcres['34']

In [ ]:
### Fit for baseline per filter -- tail end goes to zero
offsets = fit_tail(combs4, mcres_34, plot=False)

segm_mcres['offsets'] = offsets

In [ ]:
### Combine aligned data of the four segments per filter
fin_combs = comb_combs(combs_ls, [mcres_21, None, mcres_23, mcres_34], offset_fs=offsets, segment_info=[1,2,3,4])

temp_mods = comb_combs(gps_ls, [mcres_21, None, mcres_23, mcres_34], offset_fs=offsets, segment_info=[1,2,3,4])

In [ ]:
### Final model
fin_mods = {}
 ## Sigma cut first (use temporary model -- initial GP fit)
for fband in fin_combs.keys():
    if fband == 'i': excl_inds = np.where(fin_combs['i']['fitsfile'].eq('ec0915.i.030310.CT4m.x0028.a0_160305.523435_ooi_i_v1_N26.diff.fits'))[0]
    else: excl_inds = None
    sigmacut_df(fin_combs[fband], sigma=3.0, flag='sigmaclip', exclude_inds=excl_inds)

    ### Create final model
    fin_mods[fband] = optimizeGP_celer(fin_combs[fband], lc_key=None, flag='sigmaclip', exclude_inds=excl_inds)

In [ ]:
### Define phase relative to main peak
set_peak(fin_combs,fin_mods)

In [ ]:
filt_bands = ['i', 'g']
tmpl_dict = {f"fshift_{fband}_tmpl": {'init': 0.0, 'bounds': (-100, 100)}, f"ared_{fband}_tmpl": {'init': 1.0, 'bounds': (1e-4, 20.0)}, 
                 'anorm_tmpl': {'init': 1.0, 'bounds': (1e-4, 20.0)} for fband in filt_bands}

### Plots

In [ ]:
### Plot connected segments

colors4plot = ['cyan', 'green', 'red', 'magenta']

ch_fin = fin_combs
# ch_fin = temp_mods

for fband in ch_fin.keys():
    plt.title(f"{fband}-band")
    for i, s in enumerate(np.unique(ch_fin[fband]['segment'])):
        s_inds = np.where(ch_fin[fband]['segment'].eq(s) & ch_fin[fband]['sigmaclip_flag'].eq(False))[0]
        # s_inds = np.where(ch_fin[fband]['segment'].eq(s))[0]
        plt.errorbar(ch_fin[fband].loc[s_inds,'mjd'], ch_fin[fband].loc[s_inds,'muJyas2'], fmt='.', color=colors4plot[i], alpha=0.5, 
                     label=f"segment {s}", zorder=i*-1)#, yerr=ch_fin[fband].loc[s_inds,'muJyas2_err'])
    plt.axhline(0, color='k', ls='--', alpha=0.5)
    plt.show()

In [ ]:
# ### Check corner plot

# ch_samp = load_samples(f"{save_dir}/samples/1stsamples_34.h5")['samps']
ch_samp = load_samples(f"{save_dir}/samples/1stsamples_34.h5")['i']
check = corner.corner(ch_samp.values, labels=list(ch_samp.keys()), quantiles=[0.5])

In [ ]:
### Plot final segment-connected, aligned data and model

colors4plot = {'g': 'blue', 'r': 'red', 'i': 'darkred', 'z': 'chocolate'}

for fband in fin_combs.keys():

    # plt.errorbar(fin_combs[fband]['mjd'], fin_combs[fband]['muJyas2'], yerr=fin_combs[fband]['muJyas2_err'], fmt='.', ecolor='black', color='blue')
    f_inds = np.where(fin_combs[fband]['sigmaclip_flag'].eq(False))[0]
    plt.errorbar(fin_combs[fband].loc[f_inds,'mjd'], fin_combs[fband].loc[f_inds,'muJyas2'], yerr=fin_combs[fband].loc[f_inds,'muJyas2_err'], 
                 fmt='.', ecolor='black', color='blue')
    plt.errorbar(fin_mods[fband]['mjd'], fin_mods[fband]['muJyas2'], yerr=fin_mods[fband]['muJyas2_err'], fmt='-', ecolor='k', color='orange')
    # plt.ylim(-15, 75)
    plt.axhline(0, ls='--', color='k')
    plt.title(f"{fband}-band", loc='right', y=0.9, color=colors4plot[fband], fontsize=14.)
    plt.show()

# Save fits

In [ ]:
### Save fit paramaters

if not skip_1st:
    save_fitparams(mcres_1, outputfile=f"{save_dir}/fitted_params/1stparams_1.h5")
    save_fitparams(mcres_2, outputfile=f"{save_dir}/fitted_params/1stparams_2.h5")
    save_fitparams(mcres_3, outputfile=f"{save_dir}/fitted_params/1stparams_3.h5")
    save_fitparams(mcres_4, outputfile=f"{save_dir}/fitted_params/1stparams_4.h5")

In [ ]:
### Save fit segment parameters

if not skip_segm:
    save_fitparams(segm_mcres, outputfile=f"{save_dir}/fitted_params/segm_params.h5")

In [ ]:
### Save tables

save_finaldata(fin_combs, outputfile=f"{save_dir}/final_data/1st_LCdata.h5")
save_finaldata(fin_mods, outputfile=f"{save_dir}/final_data/1st_LCmodels.h5")